In [ ]:
import fishtools as ft
import warnings
import numpy as np
import os
import pandas as pd
from aicsimageio import AICSImage
import napari
from skimage import filters, morphology, measure, feature, segmentation, transform, draw, exposure, restoration
from scipy import ndimage
import re
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
def dilate_labels_by_area_fraction(labels_2d, area_fraction=0.25):
    """
    Dilate every labeled nucleus until its area increases by area_fraction.

    Parameters
    ----------
    labels_2d : np.ndarray
        2D integer label image. Background must be 0.
    area_fraction : float
        Fractional area increase:
        0.0 = no dilation
        0.25 = increase area by 25%
        1.0 = increase area by 100% (double the area)

    Returns
    -------
    dilated_labels : np.ndarray
        Dilated label image with original label IDs preserved.
    """

    if not 0 <= area_fraction <= 1:
        raise ValueError("area_fraction must be between 0 and 1.")

    if area_fraction == 0:
        return labels_2d.copy()

    footprint = morphology.disk(1)
    label_ids = np.unique(labels_2d)
    label_ids = label_ids[label_ids != 0]

    expanded_mask = np.zeros_like(labels_2d, dtype=bool)

    for label_id in label_ids:
        nucleus_mask = labels_2d == label_id

        original_area = nucleus_mask.sum()
        target_area = original_area * (1 + area_fraction)

        grown_mask = nucleus_mask.copy()

        # Dilate one pixel at a time until target area is reached
        while grown_mask.sum() < target_area:
            grown_mask = morphology.binary_dilation(
                grown_mask,
                footprint=footprint
            )

        expanded_mask |= grown_mask

    # Reassign overlapping expanded pixels to the closest original nucleus.
    # This preserves the initial nucleus IDs.
    dilated_labels = segmentation.watershed(
        image=np.zeros_like(labels_2d, dtype=float),
        markers=labels_2d,
        mask=expanded_mask
    )

    return dilated_labels

## MERVL GAG intensity quantifications
Importing images and calculating mean MERVL GAG intensity around the nucleus in a ring. The ring area is the same as the area of the nucleus (area_fraction=1).

In [ ]:
czi_list=ft.find_czi_files("/Users/agastya.singh/Desktop/MERVL EGA/36h","MERVL")

In [ ]:
# Tunable parameters
sigma = 10 # For blurring DAPI channel for nuclear segmentation
min_nucleus_size_px = 1000 # Min threshold size for nucleus. Nuclei smaller than this size will be removed.
closing_radius_px = 5 # Pixel size for closing function
area_fraction = 1.0  # 0 = no expansion; 1 = target 100% area increase
rolling_ball_radius = 50 # tune: should exceed the largest real object. Radius parameter for rolling ball background subtraction.

all_results = []

for path in czi_list[:2]:
    print(f"\nProcessing: {os.path.basename(path)}")

    viewer = None

    try:
        # Open image
        viewer = ft.view_napari(path, c=2)

        # Expected layer order from your current workflow
        layer555 = viewer.layers[1].data
        dapilayer = viewer.layers[0].data

        # Get physical pixel scale: (Z, Y, X)
        scale, ratio = ft.get_scale(viewer.layers[0])

        # ------------------------------------------------------------
        # 0. Rolling-ball background subtraction on the 555 channel
        # ------------------------------------------------------------


        layer555_bgcorr = np.empty_like(layer555, dtype=np.float32)

        for z_idx in range(layer555.shape[0]):
            plane = layer555[z_idx].astype(np.float32)
            background = restoration.rolling_ball(
                plane,
                radius=rolling_ball_radius
            )
            layer555_bgcorr[z_idx] = plane - background

        # Clip any negative residuals introduced by the subtraction
        np.clip(layer555_bgcorr, 0, None, out=layer555_bgcorr)

        # ------------------------------------------------------------
        # 1. Maximum-intensity projections
        # ------------------------------------------------------------
        layer555_mip = layer555_bgcorr.max(axis=0)
        dapilayer_mip = dapilayer.max(axis=0)

        # ------------------------------------------------------------
        # 2. Segment nuclei from the DAPI MIP
        # ------------------------------------------------------------
        dapi_blur = filters.gaussian(
            dapilayer_mip,
            sigma=sigma,
            preserve_range=True
        )

        otsu_thresh = filters.threshold_otsu(dapi_blur)
        nuclei_mask = dapi_blur > otsu_thresh

        nuclei_mask = ndimage.binary_fill_holes(nuclei_mask)

        nuclei_mask = morphology.remove_small_objects(
            nuclei_mask,
            min_size=min_nucleus_size_px
        )

        nuclei_mask = morphology.binary_closing(
            nuclei_mask,
            footprint=morphology.disk(closing_radius_px)
        )

        nuclei_labels_2d, _ = ndimage.label(nuclei_mask)

        # ------------------------------------------------------------
        # 3. If more than 2 nuclei are detected, remove the smallest
        # ------------------------------------------------------------
        # As we are only imaging 2-cell embyos, any extra nuclei detected can be removed.
        props = measure.regionprops(nuclei_labels_2d)

        if len(props) > 2:
            smallest_label = min(props, key=lambda p: p.area).label

            nuclei_labels_2d = nuclei_labels_2d.copy()
            nuclei_labels_2d[nuclei_labels_2d == smallest_label] = 0

            print(f"Removed smallest label: {smallest_label}")

        # Skip image if no nuclei remain
        if len(measure.regionprops(nuclei_labels_2d)) == 0:
            print("No nuclei remaining after segmentation.")
            continue

        # ------------------------------------------------------------
        # 4. Expand each nucleus by the chosen relative area increase
        # ------------------------------------------------------------
        nuclei_labels_dilated = dilate_labels_by_area_fraction(
            nuclei_labels_2d,
            area_fraction=area_fraction
        )

        # Ensure the MIP and label image correspond pixel-for-pixel
        assert nuclei_labels_dilated.shape == layer555_mip.shape

        # ------------------------------------------------------------
        # 5. Measure 555 signal within each expanded nucleus
        # ------------------------------------------------------------
        filename = os.path.basename(path)

        for nucleus_id in np.unique(nuclei_labels_dilated):
            if nucleus_id == 0:
                continue

            expanded_nucleus_mask = nuclei_labels_dilated == nucleus_id
            intensity_values = layer555_mip[expanded_nucleus_mask]

            all_results.append({
                "filename": filename,
                "nucleus_id": int(nucleus_id),
                "mean_555_intensity": float(np.mean(intensity_values)),
                "entropy_555": float(measure.shannon_entropy(intensity_values)),
                "expanded_nucleus_area_px": int(expanded_nucleus_mask.sum()),
                "expanded_nucleus_area_um2": float(expanded_nucleus_mask.sum() * scale[1] * scale[2])
            })

        print(
            f"Quantified {len(np.unique(nuclei_labels_dilated)) - 1} "
            "expanded nucleus/nuclei."
        )

    except Exception as e:
        print(f"FAILED: {os.path.basename(path)}")
        print(f"Reason: {e}")

    finally:
        # Close the Napari viewer before the next TIFF
        if viewer is not None:
            viewer.close()

# One row per expanded nucleus
nuclei_555_df = pd.DataFrame(all_results)

print(nuclei_555_df)

In [84]:
nuclei_555_df['condition'] = nuclei_555_df['filename'].str.split('_').str[0]
nuclei_555_df

,filename,nucleus_id,mean_555_intensity,entropy_555,expanded_nucleus_area_px,expanded_nucleus_area_um2,condition
0,36h_WT_MERVLgag-555_2024_10_31__09_44_14.czi,1,155.697377,7.061815,58416,517.641355,36h
1,36h_WT_MERVLgag-555_2024_10_31__09_44_14.czi,2,177.966888,6.998626,63059,558.784343,36h
2,36h_WT_MERVLgag-555_2024_10_31__09_47_10.czi,1,119.269087,6.862283,59910,530.880128,36h
3,36h_WT_MERVLgag-555_2024_10_31__09_47_10.czi,2,136.571332,7.071741,62447,553.361231,36h
4,36h_WT_MERVLgag-555_2024_10_31__09_50_03.czi,1,110.326852,6.755373,52510,465.306552,36h
5,36h_WT_MERVLgag-555_2024_10_31__09_50_03.czi,2,102.832882,6.635929,61250,542.754262,36h
6,36h_WT_MERVLgag-555_2024_10_31__09_52_30.czi,1,97.739458,6.452080,63199,560.024924,36h
7,36h_WT_MERVLgag-555_2024_10_31__09_52_30.czi,2,92.736584,6.396504,58550,518.828768,36h
8,36h_WT_MERVLgag-555_2024_10_31__09_54_40.czi,1,103.512575,6.682137,63698,564.446710,36h
9,36h_WT_MERVLgag-555_2024_10_31__09_54_40.czi,2,108.201985,6.748305,56638,501.885974,36h


In [85]:
nuclei_555_df.to_csv("/Users/agastya.singh/Desktop/MERVL EGA/mervl_quants_36h.tsv",sep="\t",header=True,index=False)